In [ ]:
# ==============================================================================
# VOY / MANUAL: EXPLORATORY DATA ANALYSIS & DATA AUDIT
# Datasets: customers.csv, acq_orders.csv, activity.csv
# ==============================================================================

# %% [Cell 1] Import Libraries & Configuration
from pathlib import Path
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

BASE_DIR = Path(r"C:/Users/jcstr/.dbt/voy")
SEED_DIR = BASE_DIR / "seeds"

# Visual formatting setup
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update(
    {
        "figure.figsize": (12, 6),
        "axes.titlesize": 14,
        "axes.titleweight": "bold",
        "axes.labelsize": 12,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "figure.dpi": 100,
    }
)

# %% [Cell 2] Load Datasets
customers_raw = pd.read_csv(SEED_DIR / "customers.csv")
acq_orders_raw = pd.read_csv(SEED_DIR / "acq_orders.csv")
activity_raw = pd.read_csv(SEED_DIR / "activity.csv")

print(f"customers shape:  {customers_raw.shape}")
print(f"acq_orders shape: {acq_orders_raw.shape}")
print(f"activity shape:   {activity_raw.shape}")

# %% [Cell 3] Data Overview & Quality Audit Helper
def audit_dataframe(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """Returns a structural data quality and completeness summary."""
    summary = pd.DataFrame(
        {
            "Data Type": df.dtypes,
            "Non-Null Count": df.notnull().sum(),
            "Null Count": df.isnull().sum(),
            "Null %": (df.isnull().sum() / len(df) * 100).round(2),
            "Unique Count": df.nunique(),
        }
    )
    print(f"\n--- Data Quality Audit: {name} (Rows: {len(df):,}) ---")
    print(f"Exact Duplicate Rows: {df.duplicated().sum()}")
    return summary

display(audit_dataframe(customers_raw, "customers"))
display(audit_dataframe(acq_orders_raw, "acq_orders"))
display(audit_dataframe(activity_raw, "activity"))

# %% [Cell 4] Integrity Checks & Dirty Data Profiling
# 1. Primary Key Uniqueness
print("\n--- Primary Key & Granularity Audits ---")
print(f"customers: Unique customer_id vs Total: {customers_raw['customer_id'].nunique()} / {len(customers_raw)}")
print(f"acq_orders: Unique customer_id vs Total: {acq_orders_raw['customer_id'].nunique()} / {len(acq_orders_raw)}")

# 2. Date Parsing & Temporal Logic Checks
activity_clean = activity_raw.copy()
activity_clean["from_date"] = pd.to_datetime(activity_clean["from_date"], errors="coerce")
activity_clean["to_date"] = pd.to_datetime(activity_clean["to_date"], errors="coerce")

# Check for unparseable dates
invalid_from = activity_clean["from_date"].isnull().sum()
invalid_to = (activity_clean["to_date"].isnull() & activity_raw["to_date"].notnull()).sum()
print(f"\nActivity date parsing failures: from_date={invalid_from}, to_date={invalid_to}")

# Inverted Date Check (to_date < from_date)
inverted_dates = activity_clean[activity_clean["to_date"] < activity_clean["from_date"]]
print(f"Inverted Date Ranges (to_date < from_date): {len(inverted_dates)}")

# 3. Referential Integrity (Orphan Records)
cust_ids = set(customers_raw["customer_id"])
acq_orphans = acq_orders_raw[~acq_orders_raw["customer_id"].isin(cust_ids)]
act_orphans = activity_clean[~activity_clean["customer_id"].isin(cust_ids)]
print(f"acq_orders orphans (missing from customers): {len(acq_orphans)}")
print(f"activity orphans (missing from customers):   {len(act_orphans)}")

# %% [Cell 5] Categorical Sanitisation & Descriptive Profiles
# Clean string columns
customers = customers_raw.copy()
customers["customer_country"] = customers["customer_country"].astype(str).str.strip().str.upper()

acq_orders = acq_orders_raw.copy()
acq_orders["taxonomy_business_category_group"] = (
    acq_orders["taxonomy_business_category_group"].astype(str).str.strip().str.title()
)

print("\n--- Country Distribution ---")
print(customers["customer_country"].value_counts(dropna=False))

print("\n--- Acquisition Category Distribution ---")
print(acq_orders["taxonomy_business_category_group"].value_counts(dropna=False))

# %% [Cell 6] Feature Engineering for Lifespan & Subscription Metrics
# Define active state and duration (in days)
activity_clean["is_active"] = activity_clean["to_date"].isnull()

# Compute lifespan in days (using max snapshot date for active subscriptions)
snapshot_date = activity_clean["from_date"].max()
activity_clean["effective_end_date"] = activity_clean["to_date"].fillna(snapshot_date)
activity_clean["duration_days"] = (
    activity_clean["effective_end_date"] - activity_clean["from_date"]
).dt.days

# Join customer dimension and acquisition taxonomy
master_df = activity_clean.merge(
    customers, on="customer_id", how="left"
).merge(
    acq_orders, on="customer_id", how="left"
)

# Descriptive stats on subscription duration
print("\n--- Subscription Duration (Days) Descriptive Stats ---")
display(
    master_df.groupby("taxonomy_business_category_group")["duration_days"]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.9])
    .round(1)
)

# %% [Cell 7] Visualisation 1: Churn Hazard & Duration Distribution
fig, ax = plt.subplots(figsize=(12, 5))

completed_subs = master_df[~master_df["is_active"]]
sns.histplot(
    data=completed_subs,
    x="duration_days",
    hue="taxonomy_business_category_group",
    multiple="stack",
    bins=40,
    binrange=(0, 365),
    palette="viridis",
    ax=ax,
)

ax.set_title("Subscription Lifespan Distribution at Cancellation (Days 0–365)")
ax.set_xlabel("Subscription Lifespan (Days)")
ax.set_ylabel("Cancelled Subscription Volume")
ax.axvline(30, color="crimson", linestyle="--", alpha=0.7, label="Day 30 (1st Renewal Cliff)")
ax.axvline(60, color="darkorange", linestyle="--", alpha=0.7, label="Day 60 (2nd Renewal)")
ax.legend(loc="upper right", frameon=True)
plt.tight_layout()
plt.show()

# %% [Cell 8] Visualisation 2: Monthly Cohort Retention Matrix (M0–M6)
# Build Cohort Analysis Table
master_df["cohort_month"] = master_df["from_date"].dt.to_period("M")
master_df["churn_month"] = master_df["to_date"].dt.to_period("M")

# Filter to primary historical cohorts with sufficient maturity
cohort_data = []
all_cohorts = sorted(master_df["cohort_month"].dropna().unique())

for cohort in all_cohorts[-12:]:  # Last 12 cohorts
    cohort_subs = master_df[master_df["cohort_month"] == cohort]
    total_acquired = len(cohort_subs)
    if total_acquired == 0:
        continue
    
    retention_row = {"Cohort": str(cohort), "Cohort Size": total_acquired}
    for m in range(7):
        target_month = cohort + m
        # Active in target month if started on/before and (to_date is null or ended after target month)
        active_count = len(
            cohort_subs[
                (cohort_subs["cohort_month"] <= target_month)
                & (cohort_subs["churn_month"].isnull() | (cohort_subs["churn_month"] >= target_month))
            ]
        )
        retention_row[f"M+{m}"] = round((active_count / total_acquired) * 100, 1)
    cohort_data.append(retention_row)

retention_matrix = pd.DataFrame(cohort_data).set_index("Cohort")
display(retention_matrix)

# Heatmap of Cohort Retention
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(
    retention_matrix.drop(columns=["Cohort Size"]),
    annot=True,
    fmt=".1f",
    cmap="Blues",
    cbar_kws={"label": "% Retained Subscribers"},
    ax=ax,
)
ax.set_title("Monthly Cohort Retention Rate (%)")
ax.set_xlabel("Subscription Tenure (Months)")
ax.set_ylabel("Acquisition Cohort")
plt.tight_layout()
plt.show()

# %% [Cell 9] Visualisation 3: Active Subscribers Evolution Over Time
# Generate monthly date spine to evaluate point-in-time active subscriptions
date_range = pd.date_range(
    start=master_df["from_date"].min(),
    end=master_df["from_date"].max(),
    freq="MS",
)

spine_records = []
for dt in date_range:
    for cat, group in master_df.groupby("taxonomy_business_category_group"):
        active_count = (
            (group["from_date"] <= dt)
            & (group["to_date"].isnull() | (group["to_date"] > dt))
        ).sum()
        spine_records.append({"Month": dt, "Category": cat, "Active Subscribers": active_count})

active_evolution = pd.DataFrame(spine_records)

fig, ax = plt.subplots(figsize=(12, 5))
pivot_evolution = active_evolution.pivot(index="Month", columns="Category", values="Active Subscribers").fillna(0)
pivot_evolution.plot(kind="area", stacked=True, colormap="tab10", alpha=0.85, ax=ax)

ax.set_title("Point-in-Time Active Subscribers by Category Group")
ax.set_xlabel("Date")
ax.set_ylabel("Total Active Subscriptions")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.legend(title="Category Group", loc="upper left")
plt.tight_layout()
plt.show()

# %% [Cell 10] Visualisation 4: Category vs Market Adherence Heatmap
pivot_adherence = master_df.pivot_table(
    index="customer_country",
    columns="taxonomy_business_category_group",
    values="duration_days",
    aggfunc="median",
)

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(
    pivot_adherence,
    annot=True,
    fmt=".0f",
    cmap="YlGnBu",
    cbar_kws={"label": "Median Active Days"},
    ax=ax,
)
ax.set_title("Median Subscription Duration by Country and Category (Days)")
ax.set_xlabel("Taxonomy Business Category Group")
ax.set_ylabel("Customer Country")
plt.tight_layout()
plt.show()

# %% [Cell 11] Summary Table of Derived Commercial KPIs
kpi_summary = pd.DataFrame(
    {
        "Total Customers": [customers["customer_id"].nunique()],
        "Total Subscriptions": [activity_clean["subscription_id"].nunique()],
        "Currently Active Subscriptions": [activity_clean["is_active"].sum()],
        "Active Subscription Rate (%)": [
            round((activity_clean["is_active"].sum() / len(activity_clean)) * 100, 2)
        ],
        "Overall Median Lifespan (Days)": [
            round(master_df[~master_df["is_active"]]["duration_days"].median(), 1)
        ],
        "Day 30 Early Churn Rate (%)": [
            round(
                (len(master_df[(~master_df["is_active"]) & (master_df["duration_days"] <= 31)]) / len(master_df)) * 100,
                2,
            )
        ],
    }
)

print("\n=== Executive KPI Summary ===")
display(kpi_summary.T.rename(columns={0: "Value"}))


In [ ]:
# %% [1] Company-facing analysis from dbt marts
from pathlib import Path
import duckdb
import matplotlib.ticker as mtick
from IPython.display import display

DB_PATH = Path(r"C:/Users/jcstr/.dbt/voy.duckdb")
con = duckdb.connect(str(DB_PATH), read_only=True)

as_of_date = con.execute("""
select max(date_day)
from (
    select date_day
    from main.fct_customer_daily
    group by 1
    having sum(is_active_on_date) > 0
)
""").fetchone()[0]

executive_kpis = con.execute(
    """
    with latest_day as (
        select max(date_day) as as_of_date
        from (
            select date_day
            from main.fct_customer_daily
            group by 1
            having sum(is_active_on_date) > 0
        )
    ),
    current_state as (
        select
            c.customer_id,
            coalesce(f.is_active_on_date, 0) as active_today
        from main.dim_customer c
        left join main.fct_customer_daily f
            on f.customer_id = c.customer_id
           and f.date_day = (select as_of_date from latest_day)
    )
    select
        count(*) as total_customers,
        sum(active_today) as active_customers_as_of_latest_day,
        round(1.0 * sum(active_today) / nullif(count(*), 0), 4) as active_rate_as_of_latest_day
    from current_state
    """
).df()

display(executive_kpis)

country_mix = con.execute(
    """
    with latest_day as (
        select max(date_day) as as_of_date
        from (
            select date_day
            from main.fct_customer_daily
            group by 1
            having sum(is_active_on_date) > 0
        )
    ),
    current_state as (
        select
            c.customer_id,
            c.customer_country,
            coalesce(f.is_active_on_date, 0) as active_today
        from main.dim_customer c
        left join main.fct_customer_daily f
            on f.customer_id = c.customer_id
           and f.date_day = (select as_of_date from latest_day)
    )
    select
        customer_country,
        count(*) as customers,
        sum(active_today) as active_customers,
        round(1.0 * sum(active_today) / nullif(count(*), 0), 4) as active_rate_as_of_latest_day
    from current_state
    group by 1
    order by customers desc
    """
).df()

display(country_mix)

category_mix = con.execute(
    """
    with latest_day as (
        select max(date_day) as as_of_date
        from (
            select date_day
            from main.fct_customer_daily
            group by 1
            having sum(is_active_on_date) > 0
        )
    ),
    current_state as (
        select
            c.customer_id,
            c.taxonomy_business_category_group as acquisition_category,
            coalesce(f.is_active_on_date, 0) as active_today
        from main.dim_customer c
        left join main.fct_customer_daily f
            on f.customer_id = c.customer_id
           and f.date_day = (select as_of_date from latest_day)
    )
    select
        acquisition_category,
        count(*) as customers,
        sum(active_today) as active_customers,
        round(1.0 * sum(active_today) / nullif(count(*), 0), 4) as active_rate_as_of_latest_day
    from current_state
    group by 1
    order by customers desc
    """
).df()

display(category_mix)

subscription_summary = con.execute(
    """
    select
        case when is_active then 'Open' else 'Closed' end as lifecycle_status,
        count(*) as subscription_periods,
        round(avg(duration_days), 1) as avg_duration_days,
        min(duration_days) as min_duration_days,
        max(duration_days) as max_duration_days,
        sum(activity_rows) as source_rows_folded
    from main.fct_subscription_period
    group by 1
    order by 1
    """
).df()

display(subscription_summary)

daily_trend = con.execute(
    """
    select
        date_day,
        sum(is_active_on_date) as active_customers,
        sum(is_new_customer) as new_customers,
        sum(is_inactive_on_date) as inactive_customer_days
    from main.fct_customer_daily
    group by 1
    order by 1
    """
).df()

daily_trend["date_day"] = pd.to_datetime(daily_trend["date_day"])
daily_trend["active_28d_ma"] = daily_trend["active_customers"].rolling(28, min_periods=7).mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(daily_trend["date_day"], daily_trend["active_customers"], color="#1f77b4", alpha=0.35, linewidth=1, label="Daily active customers")
ax.plot(daily_trend["date_day"], daily_trend["active_28d_ma"], color="#1f77b4", linewidth=2.5, label="28-day moving average")
ax.set_title("Daily active customers from fct_customer_daily")
ax.set_xlabel("Date")
ax.set_ylabel("Active customers")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

retention_curve = con.execute(
    """
    select cohort_month, month_index, cohort_size, active_customers, retention_rate
    from main.fct_customer_cohort_retention
    where month_index between 0 and 6
    order by 1, 2
    """
).df()

retention_curve["cohort_month"] = pd.to_datetime(retention_curve["cohort_month"])
retention_heatmap = retention_curve.pivot(index="cohort_month", columns="month_index", values="retention_rate").tail(12)

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(
    retention_heatmap,
    annot=True,
    fmt=".0%",
    cmap="Blues",
    linewidths=0.5,
    cbar_kws={"format": mtick.PercentFormatter(1.0)},
    ax=ax,
)
ax.set_title("Cohort retention heatmap from fct_customer_cohort_retention")
ax.set_xlabel("Months since cohort start")
ax.set_ylabel("Cohort month")
plt.tight_layout()
plt.show()

retention_windows = con.execute(
    """
    with windows as (
        select
            c.taxonomy_business_category_group as acquisition_category,
            datediff('day', c.first_seen_date, d.date_day) as day_index,
            d.is_active_on_date
        from main.fct_customer_daily d
        join main.dim_customer c
            on c.customer_id = d.customer_id
        where c.first_seen_date is not null
          and datediff('day', c.first_seen_date, d.date_day) in (0, 7, 25, 90)
    )
    select
        acquisition_category,
        day_index,
        count(*) as cohort_members_observed,
        sum(is_active_on_date) as active_customers
    from windows
    group by 1, 2
    order by 1, 2
    """
).df()

base_sizes = retention_windows.loc[retention_windows["day_index"] == 0, ["acquisition_category", "active_customers"]].rename(
    columns={"active_customers": "cohort_size"}
)
retention_windows = retention_windows.merge(base_sizes, on="acquisition_category", how="left")
retention_windows["retention_rate"] = retention_windows["active_customers"] / retention_windows["cohort_size"]
retention_windows_pivot = retention_windows.pivot(index="acquisition_category", columns="day_index", values="retention_rate").sort_values(0, ascending=False)

display(retention_windows_pivot)

fig, ax = plt.subplots(figsize=(14, 6))
plot_df = retention_windows.copy()
plot_df["day_label"] = plot_df["day_index"].map({0: "D0", 7: "D7", 25: "D25", 90: "D90"})
sns.barplot(data=plot_df, x="acquisition_category", y="retention_rate", hue="day_label", palette="viridis", ax=ax)
ax.set_title("Retention cliffs by acquisition category")
ax.set_xlabel("Acquisition category")
ax.set_ylabel("Retention rate")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.legend(title="Lifecycle window")
ax.tick_params(axis='x', rotation=25)
plt.tight_layout()
plt.show()

monthly_retention = con.execute(
    """
    select
        month_index,
        round(avg(retention_rate), 4) as avg_retention_rate
    from main.fct_customer_cohort_retention
    where month_index in (0, 1, 3, 6)
    group by 1
    order by 1
    """
).df()

display(monthly_retention)

print("\nCompany readout:")
print("- Marketing: acquisition category mix and D0/D7/D25/D90 retention show which channels create better long-term customers.")
print("- Product / clinical ops: the daily active trend and early retention cliffs expose onboarding friction and missed interventions.")
print("- Executive / finance: cohort retention and the active base provide a stable operating view of customer health.")
